## **공공자전거(따릉이) 대여소 정보 전처리**

- 2021 ~ 2025 서울시 따릉이 대여소 정보 종합
  
- 따릉이 이용 정보 데이터를 분석하기 위해, 각 자치구별 대여소 정보를 포함한 마스터 DB 생성
  
- 데이터셋 불러오기/전처리/병합을 수행하여 `df_merged` 생성

- `df_merged`에서 중복된 행 제거 및 데이터 최신화를 수행하여 `df_master` 생성

- `df_master` 데이터 점검, CSV 저장 (utf-8-sig)

#### **결과물: 통합_따릉이_대여소_마스터.csv**

  #### **목표 데이터 설명**
  * **대여소번호** - 각 대여소마다 부여된 고유 번호입니다.
  * **대여소명** - 버스정류장과 같이 고유 상세 지역을 이용하여 사용되는 대여소 이름입니다.
  * **자치구** - 'OO구' 형태의 서울시의 각 자치구 지역명입니다.
  * **상세주소** - 대여소가 위치한 상세 주소`(예: 서울특별시 종로구 사직로 지하130)`입니다.
  * **위도** - 대여소의 위도 정보`(예: 37.575794)`입니다.
  * **경도** - 대여소의 경도 정보입니다.

In [12]:
import pandas as pd
import numpy as np

import os
import warnings

In [13]:
# 추출할 핵심 컬럼명 정의
col_names = ['대여소번호', '대여소명', '자치구', '상세주소', '위도', '경도']

In [ ]:
# 파일 BASE_PATH 경로 정의
BASE_PATH = '../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보'

# 파일 목록 정의 (시간순 배열)
file_list = [
    '공공자전거 대여소 정보(21.01.31 기준).csv',
    '공공자전거 대여소 정보(21.06월 기준).xlsx',
    '공공자전거 대여소 정보(21.12월 기준).xlsx',
    '3. 공공자전거 대여소 정보(22.06월 기준).csv',
    '공공자전거 대여소 정보(22.12월 기준).xlsx',
    '공공자전거 대여소 정보(23.06월 기준).xlsx',
    '공공자전거 대여소 정보(23.12월 기준).xlsx',
    '공공자전거 대여소 정보(24.6월 기준).xlsx',
    '공공자전거 대여소 정보(24.12월 기준).xlsx',
    '공공자전거 대여소 정보(25.6월 기준).xlsx',
    '공공자전거 대여소 정보(25.12월 기준).xlsx'
]

# file_list의 각 요소 앞에 BASE_PATH를 붙여서 전체 파일 경로 생성
file_list = [os.path.join(BASE_PATH, file_name) for file_name in file_list]

# 결과물 파일명 정의(CSV)
f_master_name = '통합_따릉이_대여소_마스터.csv'

#### 데이터셋 불러오기
##### 각 파일 포맷에 맞추어 데이터를 불러오고, 일부 컬럼의 정제 작업을 수행합니다.

In [19]:
all_data = []

print("데이터 불러오기 작업을 시작합니다...")

for file in file_list:
    try:
        # 상단 5줄(메타데이터)을 무시하고 데이터만 로드
        if file.endswith('.csv'):
            try:
                df = pd.read_csv(file, encoding='cp949', skiprows=5, header=None)
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding='utf-8', skiprows=5, header=None)
        else:
            df = pd.read_excel(file, skiprows=5, header=None)

        # 앞쪽 6개 열만 가져와서 이름 지정
        df = df.iloc[:, :6]
        df.columns = col_names
        
        # 대여소 번호가 없는 빈 행 제거
        df = df.dropna(subset=['대여소번호'])

        # 대여소 번호 정제(Ex: 102.0 -> '102')
        df['대여소번호'] = df['대여소번호'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        df['대여소명'] = df['대여소명'].astype(str).str.strip()

        all_data.append(df)
        print(f'성공: {file} (추출된 대여소: {len(df)}개)')

    except Exception as e:
        print(f'***에러 발생*** ({file}): {e}')

print(f'총 {len(all_data)}개 데이터셋 불러오기 완료.')

데이터 불러오기 작업을 시작합니다...
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(21.01.31 기준).csv (추출된 대여소: 2154개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(21.06월 기준).xlsx (추출된 대여소: 2467개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(21.12월 기준).xlsx (추출된 대여소: 2586개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\3. 공공자전거 대여소 정보(22.06월 기준).csv (추출된 대여소: 2653개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(22.12월 기준).xlsx (추출된 대여소: 2719개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(23.06월 기준).xlsx (추출된 대여소: 2749개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(23.12월 기준).xlsx (추출된 대여소: 2762개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(24.6월 기준).xlsx (추출된 대여소: 2763개)
성공: ../../original-datasets/서울시 공공데이터/서울시 공공자전거 따릉이 대여소 정보\공공자전거 대여소 정보(24.12월 기준).xlsx (추출된 대여소: 2766개)
성공: ../../original-datasets/서울시

#### 데이터셋 병합

In [21]:
# 모든 데이터프레임 하나로 merge
df_merged = pd.concat(all_data, ignore_index=True)

print(df_merged.info())
display(df_merged.head())

<class 'pandas.DataFrame'>
RangeIndex: 29198 entries, 0 to 29197
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   대여소번호   29198 non-null  str    
 1   대여소명    29198 non-null  str    
 2   자치구     29198 non-null  str    
 3   상세주소    29198 non-null  str    
 4   위도      29086 non-null  float64
 5   경도      29086 non-null  float64
dtypes: float64(2), str(4)
memory usage: 1.3 MB
None


,대여소번호,대여소명,자치구,상세주소,위도,경도
0,301,경복궁역 7번출구 앞,종로구,서울특별시 종로구 사직로 지하130,37.575794,126.971451
1,302,경복궁역 4번출구 뒤,종로구,서울특별시 종로구 사직로 지하130,37.575947,126.974060
2,303,광화문역 1번출구 앞,종로구,서울특별시 종로구 세종대로 지하189,37.571770,126.974663
3,304,광화문역 2번출구 앞,종로구,서울특별시 종로구 세종대로 지하172,37.572113,126.977577
4,305,종로구청 옆,종로구,서울특별시 종로구 삼봉로 43,37.572582,126.978355


#### 중복 제거 및 마스터 DF 생성

In [22]:
# 중복 제거 (대여소 번호를 기준으로 가장 최신 데이터를 유지)
df_master = df_merged.drop_duplicates(subset=['대여소번호'], keep='last')

#### 결과 확인 및 저장

In [23]:
# 결과 확인
print(df_master.info())
display(df_master)

<class 'pandas.DataFrame'>
Index: 3038 entries, 101 to 29197
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   대여소번호   3038 non-null   str    
 1   대여소명    3038 non-null   str    
 2   자치구     3038 non-null   str    
 3   상세주소    3038 non-null   str    
 4   위도      3037 non-null   float64
 5   경도      3037 non-null   float64
dtypes: float64(2), str(4)
memory usage: 166.1 KB
None


,대여소번호,대여소명,자치구,상세주소,위도,경도
101,311,서울광장 옆,중구,서울특별시 중구 세종대로 지하 101,37.566612,126.977470
116,373,청구 어린이공원,중구,서울특별시 중구 다산로14길 41,37.555859,127.013855
143,444,서울역3번 출구,중구,통일로 20,37.557602,126.972580
232,868,롯데하이마트 용산점 앞,용산구,원효로208,37.537220,126.965019
345,3535,중곡사거리(국민은행),광진구,광진구 중곡동 266-5,37.559158,127.087517
...,...,...,...,...,...,...
29193,6185,가양나들목,강서구,강서구 강서로 532 동신아파트.대아아파트,37.573410,126.843452
29194,6187,마곡 119 안전센터 맞은편,강서구,강서구 강서도매시장로 72 강서농산물도매시장,37.555347,126.820724
29195,6188,금호아파트,강서구,강서구 양천로 595 염창금호타운아파트,37.556190,126.864639
29196,6189,데시앙 플렉스 지식산업센터,강서구,강서구 양천로 424 가양역 데시앙플렉스 지식산업센터,37.564484,126.848305


In [24]:
# 통합 마스터 DB 저장
df_master.to_csv(f_master_name, index=False, encoding='utf-8-sig')
print(f'{f_master_name} 파일 저장 완료.')

통합_따릉이_대여소_마스터.csv 파일 저장 완료.
